In [1]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Locate project root
project_root = Path.cwd().resolve()

if not (project_root / "data").is_dir():
    project_root = project_root.parent

data_path = project_root / "data/raw/yellow_tripdata_2025-01.parquet"

assert data_path.exists(), f"Data not found: {data_path}"

con = duckdb.connect()

print("Project root:", project_root)
print("Data:", data_path.name)
print("File size (MB):", round(data_path.stat().st_size / 1024**2, 2))

Project root: /Users/wushixundelaopo/Desktop/Urban_Mobility_Mining
Data: yellow_tripdata_2025-01.parquet
File size (MB): 56.42


In [2]:
# Section 1.1 — Dataset Schema

schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [str(data_path)]
).df()

display(schema)

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [3]:
# Section 1.2 — Basic Dataset Statistics

overview = con.execute("""
    SELECT
        COUNT(*) AS total_trips,
        MIN(tpep_pickup_datetime) AS earliest_pickup,
        MAX(tpep_pickup_datetime) AS latest_pickup,
        COUNT(DISTINCT PULocationID) AS n_pickup_zones,
        COUNT(DISTINCT DOLocationID) AS n_dropoff_zones
    FROM read_parquet(?)
""", [str(data_path)]).df()

display(overview)

,total_trips,earliest_pickup,latest_pickup,n_pickup_zones,n_dropoff_zones
0,3475226,2024-12-31 20:47:55,2025-02-01 00:00:44,261,260


In [4]:
# Section 2.1 — Missing Values and Data Quality

quality = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE tpep_pickup_datetime IS NULL
        ) AS missing_pickup_time,

        COUNT(*) FILTER (
            WHERE tpep_dropoff_datetime IS NULL
        ) AS missing_dropoff_time,

        COUNT(*) FILTER (
            WHERE PULocationID IS NULL
        ) AS missing_pickup_zone,

        COUNT(*) FILTER (
            WHERE DOLocationID IS NULL
        ) AS missing_dropoff_zone,

        COUNT(*) FILTER (
            WHERE trip_distance IS NULL
        ) AS missing_distance,

        COUNT(*) FILTER (
            WHERE tpep_dropoff_datetime < tpep_pickup_datetime
        ) AS negative_duration,

        COUNT(*) FILTER (
            WHERE tpep_dropoff_datetime = tpep_pickup_datetime
        ) AS zero_duration,

        COUNT(*) FILTER (
            WHERE trip_distance < 0
        ) AS negative_distance,

        COUNT(*) FILTER (
            WHERE trip_distance = 0
        ) AS zero_distance,

        COUNT(*) FILTER (
            WHERE PULocationID = DOLocationID
        ) AS same_origin_destination,

        COUNT(*) FILTER (
            WHERE tpep_pickup_datetime < TIMESTAMP '2025-01-01'
               OR tpep_pickup_datetime >= TIMESTAMP '2025-02-01'
        ) AS outside_january

    FROM read_parquet(?)
""", [str(data_path)]).df()

display(quality.T.rename(columns={0: "count"}))

,count
total_rows,3475226
missing_pickup_time,0
missing_dropoff_time,0
missing_pickup_zone,0
missing_dropoff_zone,0
missing_distance,0
negative_duration,124
zero_duration,1927
negative_distance,0
zero_distance,90893


In [5]:
# Section 2.2 — Investigate Zero-Distance Trips

zero_distance_audit = con.execute("""
    SELECT
        COUNT(*) AS zero_distance_trips,

        COUNT(*) FILTER (
            WHERE tpep_pickup_datetime = tpep_dropoff_datetime
        ) AS also_zero_duration,

        COUNT(*) FILTER (
            WHERE tpep_dropoff_datetime < tpep_pickup_datetime
        ) AS also_negative_duration,

        COUNT(*) FILTER (
            WHERE PULocationID = DOLocationID
        ) AS also_same_zone,

        COUNT(*) FILTER (
            WHERE trip_distance = 0
              AND PULocationID <> DOLocationID
        ) AS zero_distance_different_zones

    FROM read_parquet(?)
    WHERE trip_distance = 0
""", [str(data_path)]).df()

display(zero_distance_audit.T.rename(columns={0: "count"}))

,count
zero_distance_trips,90893
also_zero_duration,692
also_negative_duration,0
also_same_zone,30079
zero_distance_different_zones,60814


## Section 3 — Full-Year Dataset Audit (2025)

This section examines all 12 monthly NYC TLC Yellow Taxi Parquet files to assess data coverage, consistency, and quality before defining the shared preprocessing pipeline.

In [6]:
# Section 3.1 — Verify Monthly Parquet Files

monthly_files = sorted(
    (project_root / "data/raw").glob("yellow_tripdata_2025-*.parquet")
)

assert len(monthly_files) == 12, (
    f"Expected 12 monthly files, found {len(monthly_files)}"
)

file_info = []

for file in monthly_files:
    row_count = con.execute(
        "SELECT COUNT(*) FROM read_parquet(?)",
        [str(file)]
    ).fetchone()[0]

    file_info.append({
        "month": file.stem[-2:],
        "filename": file.name,
        "rows": row_count,
        "size_mb": round(file.stat().st_size / 1024**2, 2)
    })

monthly_overview = pd.DataFrame(file_info)

display(monthly_overview)

print("Total trips:", f"{monthly_overview['rows'].sum():,}")
print("Total size (MiB):", round(monthly_overview["size_mb"].sum(), 2))

,month,filename,rows,size_mb
0,01,yellow_tripdata_2025-01.parquet,3475226,56.42
1,02,yellow_tripdata_2025-02.parquet,3577543,57.55
2,03,yellow_tripdata_2025-03.parquet,4145257,66.72
3,04,yellow_tripdata_2025-04.parquet,3970553,64.23
4,05,yellow_tripdata_2025-05.parquet,4591845,74.23
5,06,yellow_tripdata_2025-06.parquet,4322960,70.14
6,07,yellow_tripdata_2025-07.parquet,3898963,63.84
7,08,yellow_tripdata_2025-08.parquet,3574091,59.41
8,09,yellow_tripdata_2025-09.parquet,4251015,69.08
9,10,yellow_tripdata_2025-10.parquet,4428699,71.78


Total trips: 48,722,602
Total size (MiB): 791.53


In [7]:
# Section 3.2 — Monthly Data Quality Audit

monthly_quality = []

for file in monthly_files:
    month = file.stem[-2:]
    start = f"2025-{month}-01"
    end = (
        f"2026-01-01" if month == "12"
        else f"2025-{int(month) + 1:02d}-01"
    )

    result = con.execute("""
        SELECT
            COUNT(*) AS total_rows,

            COUNT(*) FILTER (
                WHERE tpep_pickup_datetime IS NULL
                   OR tpep_dropoff_datetime IS NULL
                   OR PULocationID IS NULL
                   OR DOLocationID IS NULL
            ) AS missing_core_fields,

            COUNT(*) FILTER (
                WHERE tpep_dropoff_datetime < tpep_pickup_datetime
            ) AS negative_duration,

            COUNT(*) FILTER (
                WHERE tpep_dropoff_datetime = tpep_pickup_datetime
            ) AS zero_duration,

            COUNT(*) FILTER (
                WHERE trip_distance = 0
            ) AS zero_distance,

            COUNT(*) FILTER (
                WHERE PULocationID = DOLocationID
            ) AS same_zone,

            COUNT(*) FILTER (
                WHERE tpep_pickup_datetime < CAST(? AS TIMESTAMP)
                   OR tpep_pickup_datetime >= CAST(? AS TIMESTAMP)
            ) AS outside_month

        FROM read_parquet(?)
    """, [start, end, str(file)]).df()

    result.insert(0, "month", month)
    monthly_quality.append(result)

monthly_quality = pd.concat(
    monthly_quality, ignore_index=True
)

for col in [
    "missing_core_fields",
    "negative_duration",
    "zero_duration",
    "zero_distance",
    "same_zone",
    "outside_month"
]:
    monthly_quality[col + "_pct"] = (
        monthly_quality[col] /
        monthly_quality["total_rows"] * 100
    )

display(monthly_quality[
    [
        "month",
        "total_rows",
        "missing_core_fields",
        "negative_duration",
        "zero_distance_pct",
        "same_zone_pct",
        "outside_month"
    ]
].round(3))

,month,total_rows,missing_core_fields,negative_duration,zero_distance_pct,same_zone_pct,outside_month
0,01,3475226,0,124,2.615,5.194,22
1,02,3577543,0,93,2.789,4.906,31
2,03,4145257,0,81,2.502,4.725,33
3,04,3970553,0,163,2.303,4.787,7
4,05,4591845,0,101,3.073,4.573,24
5,06,4322960,0,231,3.120,4.486,20
6,07,3898963,0,1,3.175,4.539,7
7,08,3574091,0,2,2.943,4.457,17
8,09,4251015,0,0,2.928,4.544,7
9,10,4428699,0,2,2.836,4.517,13


In [8]:
# Section 3.3 — Negative Duration Investigation

negative_duration_details = []

for file in monthly_files:
    month = file.stem[-2:]

    result = con.execute("""
        SELECT
            CAST(tpep_pickup_datetime AS DATE) AS pickup_date,
            COUNT(*) AS negative_trips,
            MIN(
                date_diff(
                    'minute',
                    tpep_pickup_datetime,
                    tpep_dropoff_datetime
                )
            ) AS min_duration_minutes,
            MAX(
                date_diff(
                    'minute',
                    tpep_pickup_datetime,
                    tpep_dropoff_datetime
                )
            ) AS max_duration_minutes
        FROM read_parquet(?)
        WHERE tpep_dropoff_datetime < tpep_pickup_datetime
        GROUP BY 1
    """, [str(file)]).df()

    result.insert(0, "file_month", month)
    negative_duration_details.append(result)

negative_duration_details = pd.concat(
    negative_duration_details, ignore_index=True
)

display(
    negative_duration_details.sort_values(
        "negative_trips", ascending=False
    ).head(20)
)

,file_month,pickup_date,negative_trips,min_duration_minutes,max_duration_minutes
171,11,2025-11-02,1431,-58,-1
160,06,2025-06-20,28,0,0
158,06,2025-06-18,20,0,0
159,06,2025-06-19,18,0,0
156,06,2025-06-16,16,0,0
157,06,2025-06-17,16,0,0
151,06,2025-06-11,16,0,0
100,04,2025-04-20,15,-1,0
150,06,2025-06-10,15,0,0
96,04,2025-04-16,13,0,0


In [9]:
# Section 3.4.1 — Taxi Zone Lookup

lookup_path = project_root / "data/external/taxi_zone_lookup.csv"

assert lookup_path.exists(), f"File not found: {lookup_path}"

zones = pd.read_csv(lookup_path)

print("Number of lookup rows:", len(zones))
print("Columns:", zones.columns.tolist())

display(zones.head(10))
display(zones.isna().sum().to_frame("missing_count"))

Number of lookup rows: 265
Columns: ['LocationID', 'Borough', 'Zone', 'service_zone']


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
6,7,Queens,Astoria,Boro Zone
7,8,Queens,Astoria Park,Boro Zone
8,9,Queens,Auburndale,Boro Zone
9,10,Queens,Baisley Park,Boro Zone


,missing_count
LocationID,0
Borough,1
Zone,1
service_zone,2


In [10]:
# Section 3.4.2 — Validate Pickup and Dropoff Zone IDs

valid_ids = set(zones["LocationID"].dropna().astype(int))

zone_counts = con.execute("""
    SELECT
        'pickup' AS location_type,
        PULocationID AS location_id,
        COUNT(*) AS trip_count
    FROM read_parquet(?)
    GROUP BY 1, 2

    UNION ALL

    SELECT
        'dropoff' AS location_type,
        DOLocationID AS location_id,
        COUNT(*) AS trip_count
    FROM read_parquet(?)
    GROUP BY 1, 2
""", [str(project_root / "data/raw/yellow_tripdata_2025-*.parquet")] * 2).df()

zone_counts["in_lookup"] = zone_counts["location_id"].isin(valid_ids)

display(
    zone_counts.groupby(
        ["location_type", "in_lookup"], dropna=False
    )["trip_count"].sum().reset_index()
)

display(
    zone_counts.loc[~zone_counts["in_lookup"]]
    .sort_values("trip_count", ascending=False)
    .head(20)
)

,location_type,in_lookup,trip_count
0,dropoff,True,48722602
1,pickup,True,48722602


,location_type,location_id,trip_count,in_lookup


In [11]:
# Section 3.4.3 — Identify Special Taxi Zones

special_zones = zones[
    zones[["Borough", "Zone", "service_zone"]].isna().any(axis=1)
    |
    zones["Zone"].str.contains(
        "unknown|n/a|outside|airport",
        case=False,
        na=False
    )
]

display(special_zones.sort_values("LocationID"))

# Verify LocationID uniqueness
print("Duplicate LocationIDs:", zones["LocationID"].duplicated().sum())

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
131,132,Queens,JFK Airport,Airports
137,138,Queens,LaGuardia Airport,Airports
263,264,Unknown,NaN,NaN
264,265,NaN,Outside of NYC,NaN


Duplicate LocationIDs: 0


In [12]:
# Section 3.4.4 — Special Zone Trip Frequencies

special_ids = [1, 132, 138, 264, 265]

special_counts = con.execute("""
    SELECT
        PULocationID AS location_id,
        COUNT(*) AS pickup_count
    FROM read_parquet(?)
    WHERE PULocationID IN (1, 132, 138, 264, 265)
    GROUP BY 1
""", [str(project_root / "data/raw/yellow_tripdata_2025-*.parquet")]).df()

dropoff_counts = con.execute("""
    SELECT
        DOLocationID AS location_id,
        COUNT(*) AS dropoff_count
    FROM read_parquet(?)
    WHERE DOLocationID IN (1, 132, 138, 264, 265)
    GROUP BY 1
""", [str(project_root / "data/raw/yellow_tripdata_2025-*.parquet")]).df()

special_summary = (
    zones[zones["LocationID"].isin(special_ids)]
    [["LocationID", "Zone"]]
    .merge(
        special_counts,
        left_on="LocationID",
        right_on="location_id",
        how="left"
    )
    .drop(columns="location_id")
    .merge(
        dropoff_counts,
        left_on="LocationID",
        right_on="location_id",
        how="left"
    )
    .drop(columns="location_id")
    .fillna({"pickup_count": 0, "dropoff_count": 0})
)

special_summary["pickup_pct"] = (
    special_summary["pickup_count"] / 48722602 * 100
)

special_summary["dropoff_pct"] = (
    special_summary["dropoff_count"] / 48722602 * 100
)

display(special_summary.round(3))

,LocationID,Zone,pickup_count,dropoff_count,pickup_pct,dropoff_pct
0,1,Newark Airport,6056,94214,0.012,0.193
1,132,JFK Airport,2062456,457400,4.233,0.939
2,138,LaGuardia Airport,1291339,474334,2.650,0.974
3,264,NaN,80873,105193,0.166,0.216
4,265,Outside of NYC,22170,219073,0.046,0.450


In [13]:
# Section 3.5.1 — Full-Year Zone-Hour Demand Summary

full_year_path = str(
    project_root / "data/raw/yellow_tripdata_2025-*.parquet"
)

zone_hour_summary = con.execute("""
    WITH zone_hour AS (
        SELECT
            PULocationID AS zone_id,
            date_trunc('hour', tpep_pickup_datetime) AS hour,
            COUNT(*) AS pickup_count
        FROM read_parquet(?)
        WHERE tpep_pickup_datetime >= TIMESTAMP '2025-01-01'
          AND tpep_pickup_datetime < TIMESTAMP '2026-01-01'
          AND PULocationID NOT IN (264, 265)
        GROUP BY 1, 2
    )
    SELECT
        COUNT(*) AS observed_zone_hours,
        COUNT(DISTINCT zone_id) AS active_zones,
        MIN(pickup_count) AS min_observed_demand,
        MEDIAN(pickup_count) AS median_observed_demand,
        MAX(pickup_count) AS max_observed_demand
    FROM zone_hour
""", [full_year_path]).df()

display(zone_hour_summary)

,observed_zone_hours,active_zones,min_observed_demand,median_observed_demand,max_observed_demand
0,1415012,260,1,4.0,1300


In [14]:
# Section 3.5.2 — Zone-Level Demand Coverage

zone_coverage = con.execute("""
    WITH zone_hours AS (
        SELECT
            PULocationID AS zone_id,
            date_trunc('hour', tpep_pickup_datetime) AS hour,
            COUNT(*) AS pickup_count
        FROM read_parquet(?)
        WHERE tpep_pickup_datetime >= TIMESTAMP '2025-01-01'
          AND tpep_pickup_datetime < TIMESTAMP '2026-01-01'
          AND PULocationID NOT IN (264, 265)
        GROUP BY 1, 2
    )
    SELECT
        zone_id,
        COUNT(*) AS observed_hours,
        SUM(pickup_count) AS annual_pickups,
        ROUND(COUNT(*) * 100.0 / 8760, 2) AS observed_hour_pct
    FROM zone_hours
    GROUP BY zone_id
    ORDER BY observed_hour_pct ASC
""", [full_year_path]).df()

zone_coverage = zone_coverage.merge(
    zones[["LocationID", "Borough", "Zone"]],
    left_on="zone_id",
    right_on="LocationID",
    how="left"
).drop(columns="LocationID")

display(zone_coverage.head(15))
display(zone_coverage.tail(15))

,zone_id,observed_hours,annual_pickups,observed_hour_pct,Borough,Zone
0,99,6,7.0,0.07,Staten Island,Freshkills Park
1,105,7,7.0,0.08,Manhattan,Governor's Island/Ellis Island/Liberty Island
2,199,16,18.0,0.18,Bronx,Rikers Island
3,5,20,20.0,0.23,Staten Island,Arden Heights
4,84,26,30.0,0.30,Staten Island,Eltingville/Annadale/Prince's Bay
5,204,28,29.0,0.32,Staten Island,Rossville/Woodrow
6,44,36,40.0,0.41,Staten Island,Charleston/Tottenville
7,2,64,69.0,0.73,Queens,Jamaica Bay
8,187,69,73.0,0.79,Staten Island,Port Richmond
9,109,75,79.0,0.86,Staten Island,Great Kills


,zone_id,observed_hours,annual_pickups,observed_hour_pct,Borough,Zone
245,137,8754,534576.0,99.93,Manhattan,Kips Bay
246,132,8755,2062453.0,99.94,Queens,JFK Airport
247,163,8755,1238229.0,99.94,Manhattan,Midtown North
248,164,8756,1036929.0,99.95,Manhattan,Midtown South
249,162,8756,1508176.0,99.95,Manhattan,Midtown East
250,90,8756,737026.0,99.95,Manhattan,Flatiron
251,141,8757,1029691.0,99.97,Manhattan,Lenox Hill West
252,107,8757,923413.0,99.97,Manhattan,Gramercy
253,249,8758,1064169.0,99.98,Manhattan,West Village
254,100,8758,677196.0,99.98,Manhattan,Garment District


In [15]:
# Section 3.5.3 — Zone Demand Coverage Distribution

coverage_bins = [-0.01, 1, 10, 50, 90, 99, 100]
coverage_labels = [
    "<=1%",
    "1–10%",
    "10–50%",
    "50–90%",
    "90–99%",
    "99–100%"
]

zone_coverage["coverage_group"] = pd.cut(
    zone_coverage["observed_hour_pct"],
    bins=coverage_bins,
    labels=coverage_labels,
    include_lowest=True
)

coverage_summary = (
    zone_coverage
    .groupby("coverage_group", observed=False)
    .agg(
        n_zones=("zone_id", "count"),
        total_pickups=("annual_pickups", "sum")
    )
    .reset_index()
)

coverage_summary["pickup_share_pct"] = (
    coverage_summary["total_pickups"] /
    coverage_summary["total_pickups"].sum() * 100
)

display(coverage_summary.round(2))

,coverage_group,n_zones,total_pickups,pickup_share_pct
0,<=1%,10,372.0,0.00
1,1–10%,23,10453.0,0.02
2,10–50%,55,248877.0,0.51
3,50–90%,87,1784428.0,3.67
4,90–99%,50,9118152.0,18.75
5,99–100%,35,37457248.0,77.04


# Preliminary Data Audit — 2025 NYC Yellow Taxi Trips

**Project:** Urban Mobility Mining: Network Communities, Temporal Demand Patterns, and Taxi Demand Forecasting  
**Dataset:** NYC TLC Yellow Taxi Trip Records, January–December 2025  
**Status:** Preliminary audit completed; cleaning rules and shared data contract pending team review.

## 1. Dataset Overview

- **Raw records:** 48,722,602 trips across 12 monthly Parquet files (approximately 791.53 MiB).
- **Data structure:** The January file contains 20 columns, including pickup/dropoff timestamps, pickup/dropoff LocationIDs, trip distance, and fare-related variables.
- **January pilot:** 3,475,226 records, spanning December 31, 2024, to February 1, 2025, based on pickup timestamps.
- **Taxi Zone Lookup:** 265 records with unique LocationIDs. Every pickup and dropoff LocationID in the 2025 trip files matches an ID in the lookup table.

**Scope note:** The audit covered all 12 monthly files for record counts and selected quality indicators. Full cross-month schema consistency has not yet been independently checked.

## 2. Data Quality Findings

- No SQL NULL values were found in the four core fields: pickup datetime, dropoff datetime, pickup LocationID, and dropoff LocationID.
- The share of zero-distance trips varied from approximately **2.30% to 3.55%** across months.
- Same-zone origin–destination trips represented approximately **4.46% to 5.19%** of monthly records.
- Each monthly file contained a small number of pickup timestamps outside its nominal calendar month (7–33 records per file).
- In the January pilot, 90,893 trips had zero recorded distance, of which 60,814 had different pickup and dropoff zones.

**Time anomaly:** The November file contained 1,435 negative-duration trips; 1,431 were associated with November 2, 2025, with reported negative durations ranging approximately from −58 to −1 minutes. This pattern is consistent with the daylight-saving-time transition and requires careful timestamp handling. The exact underlying UTC times cannot be reconstructed from the current audit alone.

Zero-distance and same-zone trips should not be automatically removed without a shared decision about trip validity and the implications for pickup-demand and network analyses.

## 3. Spatial Validation

The official lookup includes several special LocationIDs:

| LocationID | Description | Annual pickups | Share of raw trips |
|---|---|---:|---:|
| 1 | Newark Airport | 6,056 | 0.012% |
| 132 | JFK Airport | 2,062,456 | 4.233% |
| 138 | LaGuardia Airport | 1,291,339 | 2.650% |
| 264 | Unknown | 80,873 | 0.166% |
| 265 | Outside of NYC | 22,170 | 0.046% |

LocationIDs 264 and 265 are candidates for exclusion from geographically localized forecasting and network analysis. Airport zones should be evaluated separately rather than treated as invalid.

**No final inclusion or exclusion rules have been applied.**

## 4. Forecasting Feasibility and Demand Sparsity

For an initial full-year aggregation, excluding pickup LocationIDs 264 and 265:

- **Active pickup zones:** 260.
- **Observed zone-hour combinations:** 1,415,012.
- **Median pickup count among observed zone-hours:** 4.
- **Nominal full panel size:** 260 × 8,760 = 2,277,600 zone-hours.
- **Unobserved zone-hour combinations:** 862,588 (approximately 37.87%), before final treatment of time-zone ambiguity and valid-zone selection.

Pickup demand is highly concentrated geographically:

| Observed-hour coverage | Number of zones | Share of pickups |
|---|---:|---:|
| ≤1% | 10 | approximately 0.00% |
| 1–10% | 23 | 0.02% |
| 10–50% | 55 | 0.51% |
| 50–90% | 87 | 3.67% |
| 90–99% | 50 | 18.75% |
| 99–100% | 35 | 77.04% |

The 35 highest-coverage zones account for approximately **77% of pickup demand**, supporting evaluation across high-, medium-, and low-demand conditions rather than relying on one overall forecasting metric.

**Important qualification:** Coverage groups were calculated using the full year for exploratory analysis only. Forecasting evaluation groups must be defined using training-period information to avoid leakage. Local timestamps also require explicit daylight-saving-time handling before constructing the final hourly panel.

## 5. Implications for Research Questions

**RQ1 — Network Communities:** The dataset contains pickup and dropoff zone identifiers suitable for constructing a weighted origin–destination network. Community structure and stability have not yet been evaluated.

**RQ2 — Temporal Patterns:** Pickup timestamps allow hourly aggregation and support subsequent investigation of daily, weekly, and seasonal patterns. Clustering results are not yet available.

**RQ3 — Forecasting:** The data supports construction of a zone-hour pickup-demand panel. The preliminary audit identifies substantial differences in demand density across zones. Predictive performance and network-feature benefits have not yet been tested.

**RQ4 — Conditional Benefits:** Demand heterogeneity motivates subgroup analysis by historical demand intensity and time of day. These comparisons remain hypotheses for future experiments.

## 6. Pending Team Decisions — Data Contract

Before implementing the shared cleaning pipeline, both team members should agree on:

1. Final LocationID inclusion rules, especially 1, 264, and 265.
2. Treatment of zero-distance, zero-duration, and negative-duration trips.
3. Time-zone and daylight-saving-time representation for hourly aggregation.
4. Exact analytical period and chronological training/validation/test split.
5. Definitions of OD network edges, self-loops, and zone-hour pickup counts.
6. The network-feature handoff format and prevention of future-information leakage.
7. Training-period definitions of low-, medium-, and high-demand evaluation groups.

**Next steps:** Finalize the shared Data Contract, implement a reproducible cleaning/aggregation pipeline, and develop the Network Mining and Forecasting modules independently using aligned data definitions.